# Unidad Modular III · Dinámica de sistemas y aplicaciones (semana 2)
## Simulación en Python y libros de Excel con dashboards

Este cuaderno acompaña la presentación y la simulación web de la semana 2. Todos los casos son **nuevos**:

| Tema de la unidad | Caso en este cuaderno | Hoja de Excel |
|---|---|---|
| Dinámica de sistemas y etapas del modelado | Racionamiento de agua de Bogotá (sistema Chingaza, 2024-2025) | `Chingaza` |
| Tipos de trayectorias | Deuda de tarjeta, tinto que se enfría, billetera digital (Bass), ducha, petróleo (Hubbert), papa (telaraña) | `Deuda`, `Tinto`, `Bass`, `Ducha`, `Petroleo`, `Papa` |
| Estabilidad y bucles | Carrera armamentista de Richardson | `Richardson` |
| Bifurcaciones | Pesquería con cuota (silla-nodo) | `Pesca` |
| Catástrofes | Lago claro o turbio (histéresis) | `Lago` |
| Características estructurales | Cadena de envejecimiento del personal de una fuerza | `Cadena` |
| Simbología | Ecuaciones estilo Vensim en cada hoja | todas |

**Cómo usarlo.** Ejecute las celdas en orden (*Entorno de ejecución → Ejecutar todas*). Cada sección simula el caso en Python, dibuja sus gráficas y agrega una hoja al libro de Excel. Al final se descarga `Dashboards_Dinamica_Sistemas_Semana2.xlsx`.

**Lo importante del Excel:** cada hoja es un modelo vivo. Las celdas amarillas con texto azul son parámetros; todo lo demás son fórmulas (método de Euler fila por fila). Cambie un parámetro en Excel y las gráficas y el tablero se actualizan solos.

In [ ]:
# Librerías (todas vienen instaladas en Google Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import LineChart, ScatterChart, BarChart, Reference, Series
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

plt.rcParams.update({"figure.figsize": (10, 4.2), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
COL = {"navy": "#0B2545", "teal": "#0E7C7B", "amber": "#F4A259", "rose": "#D1495B", "green": "#6A994E", "blue": "#3E7CB1", "violet": "#7A6C8F"}
print("Listo: numpy", np.__version__, "· pandas", pd.__version__)

### Herramientas comunes
* `euler(f, x0, T, dt)` y `rk4(f, x0, T, dt)`: integran un sistema `dx/dt = f(t, x)` y devuelven una tabla.
* Funciones de estilo para Excel: encabezados, celdas de parámetros (amarillo con texto azul), formatos y gráficas nativas.

In [ ]:
def euler(f, x0, T, dt, names=None):
    n = int(round(T / dt)); t = np.arange(n + 1) * dt; X = np.zeros((n + 1, len(x0))); X[0] = x0
    for k in range(n):
        X[k + 1] = X[k] + dt * np.asarray(f(t[k], X[k]))
    return pd.DataFrame(X, index=pd.Index(t, name="t"), columns=names or [f"x{i}" for i in range(len(x0))])

def rk4(f, x0, T, dt, names=None):
    n = int(round(T / dt)); t = np.arange(n + 1) * dt; X = np.zeros((n + 1, len(x0))); X[0] = x0
    for k in range(n):
        x = X[k]; k1 = np.asarray(f(t[k], x)); k2 = np.asarray(f(t[k] + dt / 2, x + dt / 2 * k1)); k3 = np.asarray(f(t[k] + dt / 2, x + dt / 2 * k2)); k4 = np.asarray(f(t[k] + dt, x + dt * k3))
        X[k + 1] = x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return pd.DataFrame(X, index=pd.Index(t, name="t"), columns=names or [f"x{i}" for i in range(len(x0))])

# ---------- estilo del libro de Excel ----------
F_TIT = Font(name="Arial", size=16, bold=True, color="0B2545"); F_SUB = Font(name="Arial", size=10, italic=True, color="5B6B7F")
F_HEAD = Font(name="Arial", size=10, bold=True, color="FFFFFF"); FILL_HEAD = PatternFill("solid", fgColor="13315C")
F_IN = Font(name="Arial", size=10, color="0000FF"); FILL_IN = PatternFill("solid", fgColor="FFFF00")
F_BODY = Font(name="Arial", size=10); F_BOLD = Font(name="Arial", size=10, bold=True)
F_KPI = Font(name="Arial", size=16, bold=True, color="0E7C7B"); FILL_KPI = PatternFill("solid", fgColor="EEF4F8")
THIN = Border(*(Side(style="thin", color="D5DEE8"),) * 4)

wb = Workbook(); wb.remove(wb.active)

def hoja(nombre, titulo, subtitulo):
    ws = wb.create_sheet(nombre); ws["A1"] = titulo; ws["A1"].font = F_TIT; ws["A2"] = subtitulo; ws["A2"].font = F_SUB
    ws.sheet_view.showGridLines = False; ws.column_dimensions["A"].width = 34; ws.column_dimensions["B"].width = 14
    return ws

def parametros(ws, fila, filas, titulo="Parámetros (edite las celdas amarillas)"):
    """filas: lista de (etiqueta, valor, nota). Devuelve {etiqueta: 'B#'} con referencias absolutas."""
    ws.cell(fila, 1, titulo).font = F_BOLD; ref = {}
    for i, (lab, val, nota) in enumerate(filas, start=1):
        r = fila + i; ws.cell(r, 1, lab).font = F_BODY; c = ws.cell(r, 2, val); c.font = F_IN; c.fill = FILL_IN; c.border = THIN
        if nota: ws.cell(r, 3, nota).font = F_SUB
        ref[lab] = f"$B${r}"
    return ref

def kpis(ws, fila, col, items, titulo="Indicadores"):
    ws.cell(fila, col, titulo).font = F_BOLD
    for i, (lab, formula, fmt) in enumerate(items, start=1):
        r = fila + i; a = ws.cell(r, col, lab); a.font = F_BODY; a.fill = FILL_KPI; b = ws.cell(r, col + 1, formula); b.font = F_KPI; b.fill = FILL_KPI; b.number_format = fmt
    ws.column_dimensions[get_column_letter(col)].width = 34; ws.column_dimensions[get_column_letter(col + 1)].width = 16

def encabezado(ws, fila, col, nombres, ancho=13):
    for j, n in enumerate(nombres):
        c = ws.cell(fila, col + j, n); c.font = F_HEAD; c.fill = FILL_HEAD; c.alignment = Alignment(horizontal="center", wrap_text=True)
        if ws.column_dimensions[get_column_letter(col + j)].width in (None, 13) or col + j > 2: ws.column_dimensions[get_column_letter(col + j)].width = ancho
    ws.row_dimensions[fila].height = 30

def grafica(ws, ancla, titulo, xcol, ycols, fila1, fila2, xt, yt, hoja_datos=None, ancho=17, alto=8.5, scatter=True, nombres=None):
    src = hoja_datos or ws; ch = ScatterChart() if scatter else LineChart(); ch.title = titulo; ch.style = 13; ch.x_axis.title = xt; ch.y_axis.title = yt; ch.width = ancho; ch.height = alto
    ch.x_axis.delete = False; ch.y_axis.delete = False; ch.legend.position = "b"
    xref = Reference(src, min_col=xcol, min_row=fila1, max_row=fila2); colores = ["0E7C7B", "D1495B", "F4A259", "3E7CB1", "7A6C8F", "0B2545"]
    for i, yc in enumerate(ycols):
        yref = Reference(src, min_col=yc, min_row=fila1 - 1, max_row=fila2)
        s = Series(yref, xref, title_from_data=True); s.marker.symbol = "none"; s.smooth = False
        s.graphicalProperties.line.solidFill = colores[i % 6]; s.graphicalProperties.line.width = 22000; ch.series.append(s)
    ws.add_chart(ch, ancla); return ch

def formato(ws, rango, fmt):
    for row in ws[rango]:
        for c in row: c.number_format = fmt; c.font = F_BODY
print("Herramientas listas")

---
## 1. Caso guía: el racionamiento de agua de Bogotá (sistema Chingaza)

**Datos públicos** (Alcaldía de Bogotá y EAAB):
* 11 de abril de 2024: empieza el racionamiento con Chingaza en **16,32 %**. Chingaza aporta cerca del **70 %** del agua de la ciudad.
* Consumo antes de la medida: **17,72 m³/s**; durante el racionamiento: entre 16,2 y 16,7 m³/s.
* Niveles reportados: 51,52 % (8 ago 2024), 51,69 % (19 ago), 49,82 % (29 ago) y 39,98 % (11 abr 2025, cuando se levantó la medida).
* La capacidad útil se estima en ≈ 288 millones de m³ (115 Mm³ = 39,98 %).

**Modelo didáctico (diagrama de Forrester en la presentación):**

```
Volumen   = INTEG(Afluencias − Extracción, Capacidad × 16,32 %)      [Mm³]
Afluencias = base + pulso · EXP(−((día − centro)/ancho)²)             [Mm³/día]
Extracción = 0,70 · Consumo · 0,0864                                  [Mm³/día]
Consumo    = política (sin racionamiento, fijo o adaptativo)           [m³/s]
```

Las afluencias se **calibran** con mínimos cuadrados para reproducir los cinco datos de nivel (etapa de formalización).

In [ ]:
from scipy.optimize import least_squares

CAP, N0, FRAC, C0, CR = 288.0, 16.32, 0.70, 17.72, 16.5
OBS = pd.DataFrame({"dia": [10, 129, 140, 150, 375], "nivel": [16.32, 51.52, 51.69, 49.82, 39.98], "fecha": ["11 abr 2024", "8 ago 2024", "19 ago 2024", "29 ago 2024", "11 abr 2025"]})

def afluencia(d, base, pk, centro, ancho):
    dd = ((d - centro + 182.5) % 365) - 182.5
    return base + pk * np.exp(-(dd / ancho) ** 2)

def chingaza(politica, p, dias=range(10, 400)):
    V = CAP * N0 / 100; filas = []
    for d in dias:
        nivel = V / CAP * 100
        if politica == "Sin racionamiento": c = C0
        elif politica == "Fijo": c = CR
        else: c = 15.5 if nivel < 30 else 16.3 if nivel < 50 else 17.2
        Q = afluencia(d, *p); E = FRAC * c * 0.0864
        filas.append((d, Q, c, E, V, nivel)); V = min(CAP, max(0.0, V + Q - E))
    return pd.DataFrame(filas, columns=["dia", "afluencia", "consumo", "extraccion", "volumen", "nivel"]).set_index("dia")

def residuos(p):
    sim = chingaza("Fijo", p)["nivel"]; return [sim.loc[d] - n for d, n in zip(OBS.dia, OBS.nivel)]

fit = least_squares(residuos, [0.8, 1.2, 90, 40], bounds=([0.2, 0, 30, 10], [2, 4, 200, 120]))
P_AFL = fit.x
print("Afluencia calibrada: base = %.3f, pulso = %.3f Mm³/día, centro = día %.0f, ancho = %.1f días" % tuple(P_AFL))
print("Error en los 5 datos (puntos porcentuales):", np.round(residuos(P_AFL), 2))

In [ ]:
pol = {k: chingaza(k, P_AFL) for k in ["Sin racionamiento", "Fijo", "Adaptativo"]}
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for k, c in zip(pol, [COL["rose"], COL["teal"], COL["amber"]]): ax[0].plot(pol[k].index, pol[k]["nivel"], color=c, lw=2.2, label=k)
ax[0].scatter(OBS.dia, OBS.nivel, color=COL["navy"], zorder=5, s=50, label="Datos reportados")
ax[0].axhline(20, ls="--", color="gray", lw=1); ax[0].set(title="Nivel de Chingaza (%)", xlabel="días desde el 1 de abril de 2024", ylabel="%"); ax[0].legend()
ax[1].plot(pol["Fijo"].index, pol["Fijo"]["afluencia"], color=COL["blue"], lw=2, label="Afluencias (calibradas)")
ax[1].plot(pol["Fijo"].index, pol["Fijo"]["extraccion"], color=COL["rose"], lw=2, label="Extracción con racionamiento")
ax[1].set(title="Flujos del embalse (Mm³/día)", xlabel="días"); ax[1].legend(); plt.tight_layout(); plt.show()

tabla = pd.DataFrame({k: {"Nivel mínimo (%)": v["nivel"].min(), "Días bajo 20 %": int((v["nivel"] < 20).sum()), "Nivel al año (%)": v["nivel"].iloc[-36],
                          "Ahorro vs. sin racionamiento (Mm³)": pol["Sin racionamiento"]["extraccion"].sum() - v["extraccion"].sum()} for k, v in pol.items()}).T.round(1)
tabla

**Lectura (etapa de explotación).** Sin racionamiento el embalse habría pasado unos tres meses por debajo de 20 %. La regla adaptativa (más restricción cuando el nivel está bajo) consigue un nivel al año similar al racionamiento fijo, ajustando el sacrificio a la situación.

**Advertencia (etapa de evaluación).** Las afluencias se calibraron con los mismos cinco datos con que se comparan: no es una validación independiente. Entre septiembre y marzo solo hay un dato, así que la caída simulada de febrero no está verificada.

### Hoja de Excel `Chingaza`
El modelo se escribe con **fórmulas**: cada fila es un día y el volumen del día siguiente es `MIN(Capacidad, MAX(0, Volumen + Afluencia − Extracción))` (método de Euler con paso de un día). La política se elige con una lista desplegable.

In [ ]:
ws = hoja("Chingaza", "Racionamiento de agua de Bogotá · sistema Chingaza", "Modelo didáctico calibrado con datos públicos (Alcaldía de Bogotá y EAAB, 2024-2025). Paso de Euler: 1 día.")
R = parametros(ws, 4, [("Capacidad útil (Mm³)", 288, "Estimada: 115 Mm³ = 39,98 % (EAAB, abr 2025)"), ("Nivel inicial (%)", 16.32, "11 abr 2024 (Alcaldía de Bogotá)"),
    ("Fracción de agua desde Chingaza", 0.70, "≈ 70 % (reportes públicos)"), ("Consumo sin racionamiento (m³/s)", 17.72, "Inicio de 2024 (EAAB)"),
    ("Consumo con racionamiento fijo (m³/s)", 16.5, "Promedio de reportes 16,2 a 16,7"), ("Afluencia base (Mm³/día)", round(P_AFL[0], 3), "Calibrada en este cuaderno"),
    ("Pulso estacional (Mm³/día)", round(P_AFL[1], 3), "Calibrado"), ("Centro del pulso (día)", round(P_AFL[2], 1), "Día 0 = 1 de abril de 2024"),
    ("Ancho del pulso (días)", round(P_AFL[3], 2), "Calibrado"), ("Día de inicio del racionamiento", 10, "11 de abril de 2024"),
    ("Política", "Adaptativo", "Lista: Sin racionamiento, Fijo, Adaptativo"), ("Umbral bajo (%)", 30, "Regla adaptativa"), ("Consumo si nivel < umbral bajo", 15.5, "m³/s"),
    ("Umbral medio (%)", 50, "Regla adaptativa"), ("Consumo si nivel < umbral medio", 16.3, "m³/s"), ("Consumo si nivel ≥ umbral medio", 17.2, "m³/s")])
dv = DataValidation(type="list", formula1='"Sin racionamiento,Fijo,Adaptativo"', allow_blank=False); ws.add_data_validation(dv); dv.add(R["Política"].replace("$", ""))
ws.column_dimensions["C"].width = 40
F0, NDIAS = 24, 390; FN = F0 + NDIAS - 1
encabezado(ws, F0 - 1, 1, ["Día", "Afluencia (Mm³/día)", "Consumo (m³/s)", "Extracción (Mm³/día)", "Volumen (Mm³)", "Nivel (%)", "Volumen sin racionamiento", "Nivel sin racionamiento (%)", "Extracción sin racionamiento"])
ws.column_dimensions["A"].width = 34
for i in range(NDIAS):
    r = F0 + i
    ws.cell(r, 1, f"={R['Día de inicio del racionamiento']}" if i == 0 else f"=A{r-1}+1")
    ws.cell(r, 2, f"={R['Afluencia base (Mm³/día)']}+{R['Pulso estacional (Mm³/día)']}*EXP(-(((MOD(A{r}-{R['Centro del pulso (día)']}+182.5,365)-182.5)/{R['Ancho del pulso (días)']})^2))")
    ws.cell(r, 3, f'=IF({R["Política"]}="Sin racionamiento",{R["Consumo sin racionamiento (m³/s)"]},IF({R["Política"]}="Fijo",{R["Consumo con racionamiento fijo (m³/s)"]},'
                  f'IF(F{r}<{R["Umbral bajo (%)"]},{R["Consumo si nivel < umbral bajo"]},IF(F{r}<{R["Umbral medio (%)"]},{R["Consumo si nivel < umbral medio"]},{R["Consumo si nivel ≥ umbral medio"]}))))')
    ws.cell(r, 4, f"={R['Fracción de agua desde Chingaza']}*C{r}*0.0864")
    ws.cell(r, 5, f"={R['Capacidad útil (Mm³)']}*{R['Nivel inicial (%)']}/100" if i == 0 else f"=MIN({R['Capacidad útil (Mm³)']},MAX(0,E{r-1}+B{r-1}-D{r-1}))")
    ws.cell(r, 6, f"=E{r}/{R['Capacidad útil (Mm³)']}*100")
    ws.cell(r, 7, f"=E{r}" if i == 0 else f"=MIN({R['Capacidad útil (Mm³)']},MAX(0,G{r-1}+B{r-1}-I{r-1}))")
    ws.cell(r, 8, f"=G{r}/{R['Capacidad útil (Mm³)']}*100")
    ws.cell(r, 9, f"={R['Fracción de agua desde Chingaza']}*{R['Consumo sin racionamiento (m³/s)']}*0.0864")
formato(ws, f"A{F0}:A{FN}", "0"); formato(ws, f"B{F0}:I{FN}", "0.00")
kpis(ws, 4, 5, [("Nivel mínimo con la política (%)", f"=MIN(F{F0}:F{FN})", "0.0"), ("Días por debajo de 20 % con la política", f'=COUNTIF(F{F0}:F{FN},"<20")', "0"),
    ("Días por debajo de 20 % sin racionamiento", f'=COUNTIF(H{F0}:H{FN},"<20")', "0"), ("Nivel al cabo de un año (%)", f"=INDEX(F{F0}:F{FN},355)", "0.0"),
    ("Agua ahorrada (Mm³)", f"=SUM(I{F0}:I{FN})-SUM(D{F0}:D{FN})", "0.0"), ("Nivel máximo con la política (%)", f"=MAX(F{F0}:F{FN})", "0.0")], "Indicadores del racionamiento")
encabezado(ws, 12, 5, ["Fecha", "Día", "Dato (%)", "Modelo (%)"])
for i, row in OBS.iterrows():
    r = 13 + i; ws.cell(r, 5, row.fecha).font = F_BODY; ws.cell(r, 6, int(row.dia)).font = F_BODY; ws.cell(r, 7, row.nivel).font = F_BODY
    ws.cell(r, 8, f"=INDEX($F${F0}:$F${FN},F{r}-{R['Día de inicio del racionamiento']}+1)").number_format = "0.0"
ws.cell(18, 5, "Error absoluto medio (puntos)").font = F_BOLD; ws.cell(18, 8, "=SUMPRODUCT(ABS(G13:G17-H13:H17))/5").number_format = "0.00"
ws.freeze_panes = f"B{F0}"
grafica(ws, "K3", "Nivel del embalse (%)", 1, [6, 8], F0, FN, "día (0 = 1 abr 2024)", "%")
grafica(ws, "K21", "Afluencias y extracción (Mm³/día)", 1, [2, 4], F0, FN, "día", "Mm³/día")
print("Hoja Chingaza: %d días con fórmulas" % NDIAS)

---
## 2. Tipos de trayectorias: seis modos con seis casos nuevos

| Modo | Estructura | Caso |
|---|---|---|
| Exponencial | R dominante | Deuda de tarjeta al 2,2 % mensual |
| Búsqueda de objetivo | B sin retardo | Tinto que se enfría (Newton) |
| Curva S | R que cede ante B | Billetera digital (Bass) |
| Sobrepaso y oscilación | B con retardo | La ducha (retardo de la tubería) |
| Sobrepaso y colapso | R que agota un recurso | Petróleo (Hubbert) |
| Oscilación discreta | B con retardo de siembra | Precio de la papa (telaraña) |

In [ ]:
deuda = pd.Series(2.0 * 1.022 ** np.arange(61), name="deuda")
tinto = euler(lambda t, x: [-0.045 * (x[0] - 14)], [85], 90, 1, ["temp"])
bass = euler(lambda t, x: [(0.03 + 0.38 * x[0] / 100) * (100 - x[0])], [0], 30, 0.25, ["A"])
def f_ducha(t, x, adj=2.0, D=3.0):
    v, d1, d2, d3 = x; return [(37 - d3) / adj, (v - d1) * 3 / D, (d1 - d2) * 3 / D, (d2 - d3) * 3 / D]
ducha = euler(f_ducha, [20, 20, 20, 20], 60, 0.1, ["llave", "d1", "d2", "sentida"])
def f_pet(t, x):
    R_, K = x; E = 0.1 * K * R_ / 1000; return [-E, 2.0 * E - 0.08 * K]
pet = euler(f_pet, [1000, 10], 80, 0.25, ["reservas", "capital"]); pet["extraccion"] = 0.1 * pet.capital * pet.reservas / 1000
papa = [55.0]
for _ in range(20): papa.append((100 - 10 - 0.8 * papa[-1]) / 1.0)

fig, ax = plt.subplots(2, 3, figsize=(15, 7.5))
ax[0, 0].plot(deuda, color=COL["rose"], lw=2.3); ax[0, 0].set(title="Exponencial: deuda (millones $)", xlabel="meses")
ax[0, 1].plot(tinto.index, tinto.temp, color=COL["teal"], lw=2.3); ax[0, 1].axhline(14, ls="--", color="gray"); ax[0, 1].set(title="Objetivo: tinto (°C)", xlabel="minutos")
ax[0, 2].plot(bass.index, bass.A, color=COL["blue"], lw=2.3); ax[0, 2].set(title="Curva S: billetera digital (% adoptantes)", xlabel="periodos")
ax[1, 0].plot(ducha.index, ducha.sentida, color=COL["violet"], lw=2.3); ax[1, 0].axhline(37, ls="--", color="gray"); ax[1, 0].set(title="Sobrepaso: ducha (°C)", xlabel="segundos")
ax[1, 1].plot(pet.index, pet.extraccion, color=COL["navy"], lw=2.3); ax[1, 1].set(title="Sobrepaso y colapso: extracción de petróleo", xlabel="años")
ax[1, 2].plot(papa, "o-", color=COL["amber"], lw=2); ax[1, 2].set(title="Oscilación: precio de la papa", xlabel="semestres")
plt.tight_layout(); plt.show()
print("Tiempo de duplicación de la deuda: %.1f meses (regla del 70: %.1f)" % (np.log(2) / np.log(1.022), 70 / 2.2))
print("Pico de adopción de Bass: t* = ln(q/p)/(p+q) = %.2f periodos" % (np.log(0.38 / 0.03) / 0.41))
print("Máxima temperatura sentida en la ducha: %.1f °C (sobrepaso de %.1f °C)" % (ducha.sentida.max(), ducha.sentida.max() - 37))

### Hojas de Excel de los seis modos
Cada hoja tiene parámetros editables, la tabla de Euler con fórmulas, indicadores y una gráfica. Pruebe en Excel:
* `Deuda`: agregue un pago mensual y vea cuándo deja de crecer.
* `Tinto`: cambie el paso DT a 20 y compare con la solución exacta (columna de error).
* `Bass`: suba q (boca a boca) y observe cómo se adelanta el pico.
* `Ducha`: cambie el tiempo de ajuste de 2 a 8 segundos y vea desaparecer el sobrepaso.

In [ ]:
# --- Deuda ---
ws = hoja("Deuda", "Exponencial · deuda de tarjeta de crédito", "Bucle R: deuda → intereses → deuda. Ecuación: Deuda(t+1) = Deuda(t)·(1 + tasa) − pago")
R = parametros(ws, 4, [("Deuda inicial (millones $)", 2, None), ("Tasa de interés mensual", 0.022, "Del orden de las tasas de tarjeta en Colombia"), ("Pago mensual (millones $)", 0, "Pruebe 0,05")])
ws[R["Tasa de interés mensual"].replace("$", "")].number_format = "0.0%"
encabezado(ws, 10, 1, ["Mes", "Deuda (millones $)", "Intereses del mes"])
for i in range(61):
    r = 11 + i; ws.cell(r, 1, i)
    ws.cell(r, 2, f"={R['Deuda inicial (millones $)']}" if i == 0 else f"=MAX(0,B{r-1}+C{r-1}-{R['Pago mensual (millones $)']})")
    ws.cell(r, 3, f"=B{r}*{R['Tasa de interés mensual']}")
formato(ws, "B11:C71", "0.000")
kpis(ws, 4, 5, [("Tiempo de duplicación (meses)", f"=LN(2)/LN(1+{R['Tasa de interés mensual']})", "0.0"), ("Regla del 70 (meses)", f"=70/({R['Tasa de interés mensual']}*100)", "0.0"), ("Deuda a 5 años (millones $)", "=B71", "0.00"), ("Multiplicador en 5 años", f"=B71/{R['Deuda inicial (millones $)']}", "0.00")])
grafica(ws, "H3", "Deuda sin pagos (millones $)", 1, [2], 11, 71, "mes", "millones $")

# --- Tinto ---
ws = hoja("Tinto", "Búsqueda de objetivo · el tinto que se enfría", "Ley de Newton: dT/dt = −k·(T − Tambiente). Euler: T(t+DT) = T(t) + DT·(−k·(T − Ta)). Compare con la solución exacta.")
R = parametros(ws, 4, [("Temperatura inicial (°C)", 85, None), ("Temperatura del ambiente (°C)", 14, "Bogotá"), ("k (por minuto)", 0.045, None), ("Paso DT (minutos)", 1, "Pruebe 10 y 20")])
encabezado(ws, 11, 1, ["t (min)", "Euler (°C)", "Exacta (°C)", "Error (°C)"])
for i in range(91):
    r = 12 + i
    ws.cell(r, 1, 0 if i == 0 else f"=A{r-1}+{R['Paso DT (minutos)']}")
    ws.cell(r, 2, f"={R['Temperatura inicial (°C)']}" if i == 0 else f"=B{r-1}+{R['Paso DT (minutos)']}*(-{R['k (por minuto)']}*(B{r-1}-{R['Temperatura del ambiente (°C)']}))")
    ws.cell(r, 3, f"={R['Temperatura del ambiente (°C)']}+({R['Temperatura inicial (°C)']}-{R['Temperatura del ambiente (°C)']})*EXP(-{R['k (por minuto)']}*A{r})")
    ws.cell(r, 4, f"=B{r}-C{r}")
formato(ws, "A12:D102", "0.00")
kpis(ws, 4, 6, [("Constante de tiempo τ = 1/k (min)", f"=1/{R['k (por minuto)']}", "0.0"), ("Tiempo para el 95 % de la brecha (min)", f"=3/{R['k (por minuto)']}", "0.0"), ("k·DT (estable si < 1)", f"={R['k (por minuto)']}*{R['Paso DT (minutos)']}", "0.00"), ("Error máximo de Euler (°C)", "=MAX(ABS(MIN(D12:D102)),ABS(MAX(D12:D102)))", "0.00")])
grafica(ws, "I3", "Temperatura del tinto: Euler vs. exacta", 1, [2, 3], 12, 102, "minutos", "°C")

# --- Bass ---
ws = hoja("Bass", "Curva S · adopción de una billetera digital (modelo de Bass, 1969)", "dA/dt = (p + q·A/M)·(M − A). R: boca a boca. B: saturación del mercado.")
R = parametros(ws, 4, [("Mercado potencial M (miles de usuarios)", 1000, None), ("p · innovación (publicidad)", 0.03, "Promedio reportado en la literatura"), ("q · imitación (boca a boca)", 0.38, "Promedio reportado en la literatura"), ("Paso DT (periodos)", 0.25, None)])
encabezado(ws, 11, 1, ["Periodo", "Adoptantes (miles)", "Nuevos por periodo (miles)", "Potenciales (miles)", "% del mercado"])
for i in range(121):
    r = 12 + i; M, p, q, dt = R["Mercado potencial M (miles de usuarios)"], R["p · innovación (publicidad)"], R["q · imitación (boca a boca)"], R["Paso DT (periodos)"]
    ws.cell(r, 1, 0 if i == 0 else f"=A{r-1}+{dt}")
    ws.cell(r, 2, 0 if i == 0 else f"=B{r-1}+{dt}*C{r-1}")
    ws.cell(r, 3, f"=({p}+{q}*B{r}/{M})*({M}-B{r})"); ws.cell(r, 4, f"={M}-B{r}"); ws.cell(r, 5, f"=B{r}/{M}")
formato(ws, "A12:D132", "0.00"); formato(ws, "E12:E132", "0.0%")
kpis(ws, 4, 7, [("Pico teórico t* = ln(q/p)/(p+q)", f"=LN({q}/{p})/({p}+{q})", "0.00"), ("Pico simulado (periodo)", "=INDEX(A12:A132,MATCH(MAX(C12:C132),C12:C132,0))", "0.00"),
    ("Máximo de nuevos por periodo (miles)", "=MAX(C12:C132)", "0.0"), ("Adopción al periodo 10", "=INDEX(E12:E132,41)", "0.0%")])
grafica(ws, "J3", "Adoptantes acumulados (miles)", 1, [2], 12, 132, "periodo", "miles")
grafica(ws, "J21", "Nuevos adoptantes por periodo (miles)", 1, [3], 12, 132, "periodo", "miles")

# --- Ducha ---
ws = hoja("Ducha", "Sobrepaso · la ducha con retardo en la tubería", "La persona mueve la llave según lo que siente; el agua tarda en llegar (retardo de tercer orden, como DELAY3 de Vensim).")
R = parametros(ws, 4, [("Temperatura deseada (°C)", 37, None), ("Tiempo de ajuste (s)", 2, "Pruebe 8"), ("Retardo de la tubería (s)", 3, None), ("Temperatura inicial (°C)", 20, None), ("Paso DT (s)", 0.1, None)])
encabezado(ws, 12, 1, ["t (s)", "Temperatura en la llave", "Etapa 1", "Etapa 2", "Temperatura sentida"])
Td, aj, Dl, T0, dt = R["Temperatura deseada (°C)"], R["Tiempo de ajuste (s)"], R["Retardo de la tubería (s)"], R["Temperatura inicial (°C)"], R["Paso DT (s)"]
for i in range(601):
    r = 13 + i
    if i == 0:
        ws.cell(r, 1, 0); [ws.cell(r, c, f"={T0}") for c in range(2, 6)]
    else:
        ws.cell(r, 1, f"=A{r-1}+{dt}"); ws.cell(r, 2, f"=B{r-1}+{dt}*({Td}-E{r-1})/{aj}")
        ws.cell(r, 3, f"=C{r-1}+{dt}*(B{r-1}-C{r-1})*3/{Dl}"); ws.cell(r, 4, f"=D{r-1}+{dt}*(C{r-1}-D{r-1})*3/{Dl}"); ws.cell(r, 5, f"=E{r-1}+{dt}*(D{r-1}-E{r-1})*3/{Dl}")
formato(ws, "A13:E613", "0.00")
kpis(ws, 4, 7, [("Temperatura máxima sentida (°C)", "=MAX(E13:E613)", "0.0"), ("Sobrepaso (°C)", f"=MAX(E13:E613)-{Td}", "0.0"), ("Momento del máximo (s)", "=INDEX(A13:A613,MATCH(MAX(E13:E613),E13:E613,0))", "0.0")])
grafica(ws, "J3", "Temperatura sentida (°C)", 1, [5, 2], 13, 613, "segundos", "°C")

# --- Petróleo ---
ws = hoja("Petroleo", "Sobrepaso y colapso · petróleo de un campo (inspirado en Hubbert, 1956)", "Extracción = eficiencia·Capital·Reservas/Reservas iniciales. R: capital → extracción → ingresos → capital. B: agotamiento.")
R = parametros(ws, 4, [("Reservas iniciales", 1000, "millones de barriles"), ("Capital inicial", 10, "unidades de capital"), ("Eficiencia de extracción", 0.1, None), ("Reinversión por unidad extraída", 2.0, None), ("Depreciación anual", 0.08, None), ("Paso DT (años)", 0.25, None)])
encabezado(ws, 13, 1, ["Año", "Reservas", "Capital", "Extracción anual"])
R0, K0, ef, ri, dp, dt = (R[k] for k in ["Reservas iniciales", "Capital inicial", "Eficiencia de extracción", "Reinversión por unidad extraída", "Depreciación anual", "Paso DT (años)"])
for i in range(321):
    r = 14 + i
    ws.cell(r, 1, 0 if i == 0 else f"=A{r-1}+{dt}")
    ws.cell(r, 2, f"={R0}" if i == 0 else f"=MAX(0,B{r-1}-{dt}*D{r-1})")
    ws.cell(r, 3, f"={K0}" if i == 0 else f"=C{r-1}+{dt}*({ri}*D{r-1}-{dp}*C{r-1})")
    ws.cell(r, 4, f"={ef}*C{r}*B{r}/{R0}")
formato(ws, "A14:D334", "0.00")
kpis(ws, 4, 6, [("Extracción máxima", "=MAX(D14:D334)", "0.0"), ("Año del pico", "=INDEX(A14:A334,MATCH(MAX(D14:D334),D14:D334,0))", "0.0"), ("Reservas en el pico (% de las iniciales)", f"=INDEX(B14:B334,MATCH(MAX(D14:D334),D14:D334,0))/{R0}", "0.0%"), ("Reservas que quedan al final", "=B334", "0.0")])
grafica(ws, "I3", "Extracción anual", 1, [4], 14, 334, "años", "extracción")
grafica(ws, "I21", "Reservas y capital", 1, [2, 3], 14, 334, "años", "unidades")

# --- Papa ---
ws = hoja("Papa", "Oscilación · telaraña del precio de la papa (Ezekiel, 1938)", "Demanda: D = a − b·P. Oferta del semestre: S = c + d·P(anterior). Precio que vacía el mercado: P = (a − c − d·P anterior)/b")
R = parametros(ws, 4, [("a (demanda máxima)", 100, None), ("b (sensibilidad de la demanda)", 1.0, None), ("c (oferta mínima)", 10, None), ("d (reacción de la siembra)", 0.8, "Pruebe 1,15"), ("Precio inicial", 55, None)])
encabezado(ws, 12, 1, ["Semestre", "Precio", "Oferta (siembra)", "Demanda"])
a_, b_, c_, d_, P0 = (R[k] for k in ["a (demanda máxima)", "b (sensibilidad de la demanda)", "c (oferta mínima)", "d (reacción de la siembra)", "Precio inicial"])
for i in range(21):
    r = 13 + i; ws.cell(r, 1, i)
    ws.cell(r, 2, f"={P0}" if i == 0 else f"=({a_}-{c_}-{d_}*B{r-1})/{b_}")
    if i > 0: ws.cell(r, 3, f"={c_}+{d_}*B{r-1}")
    ws.cell(r, 4, f"={a_}-{b_}*B{r}")
formato(ws, "B13:D33", "0.00")
kpis(ws, 4, 6, [("Precio de equilibrio (a − c)/(b + d)", f"=({a_}-{c_})/({b_}+{d_})", "0.00"), ("Razón d/b", f"={d_}/{b_}", "0.00"), ("Diagnóstico", f'=IF({d_}<{b_},"Oscilación amortiguada",IF({d_}={b_},"Oscilación constante","Oscilación creciente"))', "@")])
grafica(ws, "I3", "Precio de la papa por semestre", 1, [2], 13, 33, "semestre", "precio")
print("Hojas de modos listas")

---
## 3. Estabilidad y bucles: la carrera armamentista de Richardson

$$\frac{dx}{dt} = k\,y - a\,x + g, \qquad \frac{dy}{dt} = l\,x - b\,y + h$$

Los bucles de **escalada** (k, l) son de refuerzo; los de **fatiga** (a, b) son de balance. El jacobiano es $\begin{pmatrix}-a & k\\ l & -b\end{pmatrix}$ y el equilibrio es estable si $a\,b > k\,l$.

In [ ]:
def richardson(k, l, a, b, g=1.0, h=1.0, T=40, dt=0.05):
    return rk4(lambda t, z: [k * z[1] - a * z[0] + g, l * z[0] - b * z[1] + h], [2, 1], T, dt, ["X", "Y"])
casos = {"Estable (k=0,4; l=0,5)": (0.4, 0.5, 0.6, 0.7), "Escalada (k=0,8; l=0,9)": (0.8, 0.9, 0.6, 0.7)}
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
for axi, (nom, (k, l, a, b)) in zip(ax, casos.items()):
    d = richardson(k, l, a, b, T=40 if "Estable" in nom else 12); axi.plot(d.index, d.X, lw=2.3, color=COL["teal"], label="País X"); axi.plot(d.index, d.Y, lw=2.3, color=COL["amber"], label="País Y")
    ev = np.linalg.eigvals([[-a, k], [l, -b]]); axi.set(title=f"{nom}\nautovalores: {ev[0]:.2f} y {ev[1]:.2f} · a·b − k·l = {a*b-k*l:.2f}", xlabel="años", ylabel="armas"); axi.legend()
plt.tight_layout(); plt.show()

In [ ]:
ws = hoja("Richardson", "Equilibrio y estabilidad · carrera armamentista de Richardson", "dx/dt = k·y − a·x + g ; dy/dt = l·x − b·y + h. Estable si a·b > k·l.")
R = parametros(ws, 4, [("k: reacción de X a las armas de Y", 0.4, "Pruebe 0,8"), ("l: reacción de Y a las armas de X", 0.5, "Pruebe 0,9"), ("a: fatiga o costo de X", 0.6, None), ("b: fatiga o costo de Y", 0.7, None),
    ("g: agravios de X", 1, None), ("h: agravios de Y", 1, None), ("Armas iniciales de X", 2, None), ("Armas iniciales de Y", 1, None), ("Paso DT (años)", 0.1, None)])
k_, l_, a_, b_, g_, h_, x0, y0, dt = (R[k] for k in list(R))
encabezado(ws, 16, 1, ["Año", "Armas de X", "Armas de Y"])
for i in range(301):
    r = 17 + i
    ws.cell(r, 1, 0 if i == 0 else f"=A{r-1}+{dt}")
    ws.cell(r, 2, f"={x0}" if i == 0 else f"=B{r-1}+{dt}*({k_}*C{r-1}-{a_}*B{r-1}+{g_})")
    ws.cell(r, 3, f"={y0}" if i == 0 else f"=C{r-1}+{dt}*({l_}*B{r-1}-{b_}*C{r-1}+{h_})")
formato(ws, "A17:C317", "0.00")
kpis(ws, 4, 5, [("a·b − k·l (estable si > 0)", f"={a_}*{b_}-{k_}*{l_}", "0.000"), ("Autovalor 1", f"=(-({a_}+{b_})+SQRT(({a_}-{b_})^2+4*{k_}*{l_}))/2", "0.000"), ("Autovalor 2", f"=(-({a_}+{b_})-SQRT(({a_}-{b_})^2+4*{k_}*{l_}))/2", "0.000"),
    ("Diagnóstico", f'=IF({a_}*{b_}>{k_}*{l_},"Equilibrio estable","Escalada (punto de silla)")', "@"), ("Equilibrio de X (si es estable)", f"=IF({a_}*{b_}>{k_}*{l_},({b_}*{g_}+{k_}*{h_})/({a_}*{b_}-{k_}*{l_}),\"no aplica\")", "0.00"), ("Armas de X a los 30 años", "=B317", "0.0")])
grafica(ws, "H12", "Armas de cada país", 1, [2, 3], 17, 317, "años", "armas")
print("Hoja Richardson lista")

---
## 4. Bifurcación silla-nodo: la pesquería con cuota

$$\frac{dN}{dt} = r\,N\left(1-\frac{N}{K}\right) - H$$

Equilibrios: $N^* = \frac{K}{2}\left(1 \pm \sqrt{1 - \frac{4H}{rK}}\right)$. El de arriba es estable y el de abajo es un umbral inestable. En $H = rK/4$ (rendimiento máximo sostenible) chocan y desaparecen.

In [ ]:
r_, K_ = 0.5, 1000
H = np.linspace(0, 140, 281); disc = 1 - 4 * H / (r_ * K_)
up = np.where(disc >= 0, K_ / 2 * (1 + np.sqrt(np.clip(disc, 0, None))), np.nan); dn = np.where(disc >= 0, K_ / 2 * (1 - np.sqrt(np.clip(disc, 0, None))), np.nan)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
ax[0].plot(H, up, color=COL["teal"], lw=2.5, label="estable"); ax[0].plot(H, dn, "--", color=COL["rose"], lw=2.2, label="umbral inestable"); ax[0].axvline(r_ * K_ / 4, color="gray", ls=":")
ax[0].set(title="Diagrama de bifurcación", xlabel="cuota H (t/año)", ylabel="N* (t)"); ax[0].legend()
for Hq, c in [(100, COL["teal"]), (120, COL["amber"]), (130, COL["rose"])]:
    d = euler(lambda t, x: [r_ * x[0] * (1 - x[0] / K_) - Hq if x[0] > 0 else 0.0], [800], 80, 0.05, ["N"]); ax[1].plot(d.index, d.N.clip(lower=0), color=c, lw=2.2, label=f"H = {Hq}")
ax[1].set(title="Población con distintas cuotas", xlabel="años", ylabel="t"); ax[1].legend(); plt.tight_layout(); plt.show()
print("Rendimiento máximo sostenible rK/4 =", r_ * K_ / 4, "t/año")

In [ ]:
ws = hoja("Pesca", "Bifurcación silla-nodo · pesquería con cuota fija", "dN/dt = r·N·(1 − N/K) − H. Si H > r·K/4 no hay equilibrio y la población colapsa.")
R = parametros(ws, 4, [("r: tasa de crecimiento (1/año)", 0.5, None), ("K: capacidad de carga (t)", 1000, None), ("H: cuota anual (t/año)", 100, "Pruebe 120 y 130"), ("Población inicial (t)", 800, None), ("Paso DT (años)", 0.1, None)])
rr, KK, HH, N0_, dt = (R[k] for k in list(R))
encabezado(ws, 12, 1, ["Año", "Población (t)", "Crecimiento natural", "Captura"])
for i in range(801):
    r = 13 + i
    ws.cell(r, 1, 0 if i == 0 else f"=A{r-1}+{dt}")
    ws.cell(r, 2, f"={N0_}" if i == 0 else f"=MAX(0,B{r-1}+{dt}*(C{r-1}-D{r-1}))")
    ws.cell(r, 3, f"={rr}*B{r}*(1-B{r}/{KK})"); ws.cell(r, 4, f"=IF(B{r}>0,{HH},0)")
formato(ws, "A13:D813", "0.0")
encabezado(ws, 12, 6, ["Cuota H", "N* estable", "N* umbral"])
for i in range(26):
    r = 13 + i; ws.cell(r, 6, f"=ROUND({rr}*{KK}/4*{i}/25,2)")
    ws.cell(r, 7, f"={KK}/2*(1+SQRT(MAX(0,1-4*F{r}/({rr}*{KK}))))"); ws.cell(r, 8, f"={KK}/2*(1-SQRT(MAX(0,1-4*F{r}/({rr}*{KK}))))")
formato(ws, "F13:H38", "0.0")
kpis(ws, 4, 10, [("Rendimiento máximo sostenible rK/4", f"={rr}*{KK}/4", "0.0"), ("Discriminante 1 − 4H/(rK)", f"=1-4*{HH}/({rr}*{KK})", "0.000"),
    ("Equilibrio estable N*", f'=IF(1-4*{HH}/({rr}*{KK})>=0,{KK}/2*(1+SQRT(1-4*{HH}/({rr}*{KK}))),"no existe")', "0.0"), ("Umbral de colapso", f'=IF(1-4*{HH}/({rr}*{KK})>=0,{KK}/2*(1-SQRT(1-4*{HH}/({rr}*{KK}))),"no existe")', "0.0"),
    ("Población a los 80 años", "=B813", "0.0"), ("Diagnóstico", f'=IF({HH}>{rr}*{KK}/4,"Colapso: cuota mayor que el máximo sostenible","Sostenible si N supera el umbral")', "@")])
grafica(ws, "M3", "Población (t)", 1, [2], 13, 813, "años", "t")
grafica(ws, "M21", "Diagrama de bifurcación", 6, [7, 8], 13, 38, "cuota H", "N*")
print("Hoja Pesca lista")

---
## 5. Catástrofe e histéresis: el lago claro o turbio

$$\frac{dP}{dt} = a - b\,P + \frac{P^{8}}{1 + P^{8}}$$

(Carpenter, Ludwig y Brock, 1999). P = fósforo en el agua; a = aporte externo; b = pérdida. Con b = 0,8 el lago claro colapsa en a ≈ 0,51 y solo vuelve si a baja a ≈ 0,14. Abajo se barre a lentamente hacia arriba y hacia abajo para ver la **histéresis**.

In [ ]:
b_l, q_l = 0.8, 8
Pg = np.linspace(0.001, 2.5, 600); ag = b_l * Pg - Pg ** q_l / (1 + Pg ** q_l); est = b_l - q_l * Pg ** (q_l - 1) / (1 + Pg ** q_l) ** 2 > 0
T_, dt_ = 400, 0.1; n_ = int(T_ / dt_); tt = np.arange(n_ + 1) * dt_; a_t = 0.05 + 0.6 * (1 - np.abs(1 - 2 * tt / T_)); Pt = np.zeros(n_ + 1); Pt[0] = 0.06
for k in range(n_): Pt[k + 1] = Pt[k] + dt_ * (a_t[k] - b_l * Pt[k] + Pt[k] ** q_l / (1 + Pt[k] ** q_l))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
ax[0].plot(np.where(est, ag, np.nan), Pg, color=COL["teal"], lw=2.5, label="estable"); ax[0].plot(np.where(~est, ag, np.nan), Pg, "--", color=COL["rose"], lw=2, label="inestable")
ax[0].set(xlim=(0, 0.8), title="Equilibrios del lago", xlabel="aporte a", ylabel="fósforo P*"); ax[0].legend()
mid = n_ // 2; ax[1].plot(a_t[:mid], Pt[:mid], color=COL["rose"], lw=2.2, label="subiendo a"); ax[1].plot(a_t[mid:], Pt[mid:], color=COL["blue"], lw=2.2, label="bajando a")
ax[1].set(title="Histéresis: ida y regreso son distintos", xlabel="aporte a", ylabel="fósforo P"); ax[1].legend(); plt.tight_layout(); plt.show()
lo = (Pg < 1); print("Salto hacia el estado turbio en a ≈ %.3f; regreso al estado claro en a ≈ %.3f" % (ag[lo].max(), ag[~lo].min()))

In [ ]:
ws = hoja("Lago", "Catástrofe e histéresis · el lago claro o turbio", "dP/dt = a − b·P + P^q/(1 + P^q) (Carpenter et al., 1999). Primero los equilibrios; luego un barrido lento de a hacia arriba y hacia abajo.")
R = parametros(ws, 4, [("b: pérdida de fósforo", 0.8, "Pruebe 0,6: regreso imposible"), ("q: no linealidad del reciclaje", 8, None), ("a mínimo del barrido", 0.05, None), ("a máximo del barrido", 0.65, None), ("Duración del barrido", 400, None), ("Paso DT", 0.2, None), ("Fósforo inicial", 0.06, None)])
bL, qL, amin, amax, TT, dtL, P0L = (R[k] for k in list(R))
encabezado(ws, 14, 1, ["P (equilibrio)", "a que lo produce", "Pendiente de retorno", "Estabilidad"])
for i in range(250):
    r = 15 + i; ws.cell(r, 1, round(0.01 * (i + 1), 2))
    ws.cell(r, 2, f"={bL}*A{r}-A{r}^{qL}/(1+A{r}^{qL})"); ws.cell(r, 3, f"={bL}-{qL}*A{r}^({qL}-1)/(1+A{r}^{qL})^2"); ws.cell(r, 4, f'=IF(C{r}>0,"estable","inestable")')
formato(ws, "A15:C264", "0.000")
encabezado(ws, 14, 6, ["t", "Aporte a(t)", "Fósforo P(t)"])
for i in range(2001):
    r = 15 + i
    ws.cell(r, 6, 0 if i == 0 else f"=F{r-1}+{dtL}")
    ws.cell(r, 7, f"={amin}+({amax}-{amin})*(1-ABS(1-2*F{r}/{TT}))")
    ws.cell(r, 8, f"={P0L}" if i == 0 else f"=MAX(0,H{r-1}+{dtL}*(G{r-1}-{bL}*H{r-1}+H{r-1}^{qL}/(1+H{r-1}^{qL})))")
formato(ws, "F15:H2015", "0.000")
kpis(ws, 4, 10, [("Salto a turbio: a máximo de la rama baja", "=MAX(B15:B113)", "0.000"), ("Regreso a claro: a mínimo de la rama alta", "=MIN(B114:B264)", "0.000"), ("Ancho de la histéresis", "=K5-K6", "0.000"),
    ("Diagnóstico", '=IF(K6<0,"Irreversible: no se puede volver",IF(K6<K5,"Histéresis reversible","Sin catástrofe"))', "@")])
grafica(ws, "M3", "Equilibrios: P* según el aporte a", 2, [1], 15, 264, "aporte a", "P*")
grafica(ws, "M21", "Histéresis en el barrido (P vs. a)", 7, [8], 15, 2015, "aporte a", "P")
print("Hoja Lago lista")

---
## 6. Característica estructural: cadena de envejecimiento del personal

Aspirantes → Cadetes (formación 4 años) → Oficiales (servicio 20 años) → Retirados. Entre los años 5 y 8 la incorporación cae al 40 %. El «hueco» recorre la cadena con retardos.

In [ ]:
def cadena(T=40, dt=0.05, recorte=0.4):
    def f(t, x):
        A, C, O, Rt = x; rec = 300 * (recorte if 5 <= t < 8 else 1.0)
        return [rec - A / 1.0, 0.8 * A / 1.0 - C / 4.0, C / 4.0 - O / 20.0, O / 20.0 - Rt / 15.0]
    return euler(f, [300, 960, 4800, 3600], T, dt, ["Aspirantes", "Cadetes", "Oficiales", "Retirados"])
cad = cadena(); cad["Graduados por año"] = cad.Cadetes / 4
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
ax[0].plot(cad.index, cad["Graduados por año"], color=COL["amber"], lw=2.3); ax[0].axvspan(5, 8, color="gray", alpha=0.15); ax[0].set(title="Graduados por año", xlabel="años")
ax[1].plot(cad.index, cad.Oficiales, color=COL["blue"], lw=2.3); ax[1].axvspan(5, 8, color="gray", alpha=0.15); ax[1].set(title="Oficiales activos", xlabel="años")
plt.tight_layout(); plt.show()
print("Mínimo de oficiales: %.0f en el año %.1f" % (cad.Oficiales.min(), cad.Oficiales.idxmin()))

In [ ]:
ws = hoja("Cadena", "Estructura · cadena de envejecimiento del personal de una fuerza", "Niveles en serie: Aspirantes → Cadetes → Oficiales → Retirados. Cifras ilustrativas.")
R = parametros(ws, 4, [("Incorporación normal (personas/año)", 300, None), ("Fracción que pasa a cadete", 0.8, None), ("Tiempo como aspirante (años)", 1, None), ("Duración de la formación (años)", 4, None),
    ("Años de servicio", 20, None), ("Permanencia en el registro de retirados (años)", 15, None), ("Recorte de incorporación (fracción)", 0.4, "1 = sin recorte"), ("Inicio del recorte (año)", 5, None), ("Fin del recorte (año)", 8, None), ("Paso DT (años)", 0.1, None)])
inc, fr, ta, tf, ts, tr, rc, ri, rf, dt = (R[k] for k in list(R))
encabezado(ws, 17, 1, ["Año", "Incorporación", "Aspirantes", "Cadetes", "Oficiales", "Retirados", "Graduados por año"])
for i in range(401):
    r = 18 + i
    ws.cell(r, 1, 0 if i == 0 else f"=A{r-1}+{dt}")
    ws.cell(r, 2, f"={inc}*IF(AND(A{r}>={ri},A{r}<{rf}),{rc},1)")
    if i == 0:
        for c, v in zip(range(3, 7), [300, 960, 4800, 3600]): ws.cell(r, c, v)
    else:
        ws.cell(r, 3, f"=C{r-1}+{dt}*(B{r-1}-C{r-1}/{ta})"); ws.cell(r, 4, f"=D{r-1}+{dt}*({fr}*C{r-1}/{ta}-D{r-1}/{tf})")
        ws.cell(r, 5, f"=E{r-1}+{dt}*(D{r-1}/{tf}-E{r-1}/{ts})"); ws.cell(r, 6, f"=F{r-1}+{dt}*(E{r-1}/{ts}-F{r-1}/{tr})")
    ws.cell(r, 7, f"=D{r}/{tf}")
formato(ws, "A18:G418", "0.0")
kpis(ws, 4, 9, [("Oficiales mínimos", "=MIN(E18:E418)", "0"), ("Año del mínimo de oficiales", "=INDEX(A18:A418,MATCH(MIN(E18:E418),E18:E418,0))", "0.0"), ("Graduados mínimos por año", "=MIN(G18:G418)", "0"),
    ("Año del mínimo de graduados", "=INDEX(A18:A418,MATCH(MIN(G18:G418),G18:G418,0))", "0.0"), ("Pérdida máxima de oficiales (%)", "=1-MIN(E18:E418)/E18", "0.0%")])
grafica(ws, "L3", "Graduados por año", 1, [7], 18, 418, "años", "personas/año")
grafica(ws, "L21", "Oficiales activos", 1, [5], 18, 418, "años", "personas")
print("Hoja Cadena lista")

---
## 7. Tablero (dashboard) e índice
La hoja `Tablero` reúne los indicadores clave con fórmulas que apuntan a cada hoja; la hoja `Inicio` explica el libro. Al cambiar un parámetro en cualquier hoja, el tablero se actualiza.

In [ ]:
ws = wb.create_sheet("Tablero", 0); ws.sheet_view.showGridLines = False
ws["A1"] = "Tablero · Dinámica de sistemas, semana 2"; ws["A1"].font = F_TIT
ws["A2"] = "Indicadores vivos: cada cifra es una fórmula que apunta a la hoja del caso."; ws["A2"].font = F_SUB
tiles = [("Chingaza · días bajo 20 % con la política", "=Chingaza!F6", "0"), ("Chingaza · días bajo 20 % sin racionamiento", "=Chingaza!F7", "0"), ("Chingaza · agua ahorrada (Mm³)", "=Chingaza!F9", "0.0"),
    ("Deuda · tiempo de duplicación (meses)", "=Deuda!F5", "0.0"), ("Tinto · constante de tiempo (min)", "=Tinto!G5", "0.0"), ("Bass · pico simulado (periodo)", "=Bass!H6", "0.0"),
    ("Ducha · sobrepaso (°C)", "=Ducha!H6", "0.0"), ("Petróleo · año del pico", "=Petroleo!G6", "0.0"), ("Papa · diagnóstico", "=Papa!G7", "@"),
    ("Richardson · diagnóstico", "=Richardson!F8", "@"), ("Pesca · máximo sostenible (t/año)", "=Pesca!K5", "0.0"), ("Pesca · diagnóstico", "=Pesca!K10", "@"),
    ("Lago · salto a turbio (a)", "=Lago!K5", "0.000"), ("Lago · regreso a claro (a)", "=Lago!K6", "0.000"), ("Cadena · oficiales mínimos", "=Cadena!J5", "0")]
for i, (lab, f, fmt) in enumerate(tiles):
    r = 4 + (i // 3) * 3; c = 1 + (i % 3) * 3
    a = ws.cell(r, c, lab); a.font = Font(name="Arial", size=9, color="5B6B7F"); a.fill = FILL_KPI
    b = ws.cell(r + 1, c, f); b.font = Font(name="Arial", size=18, bold=True, color="0E7C7B"); b.fill = FILL_KPI; b.number_format = fmt
    ws.merge_cells(start_row=r, start_column=c, end_row=r, end_column=c + 1); ws.merge_cells(start_row=r + 1, start_column=c, end_row=r + 1, end_column=c + 1)
for col in range(1, 10): ws.column_dimensions[get_column_letter(col)].width = 20 if col % 3 else 3
grafica(ws, "A20", "Chingaza: nivel del embalse (%)", 1, [6, 8], 24, 24 + 389, "día", "%", hoja_datos=wb["Chingaza"], ancho=16, alto=7.5)
grafica(ws, "F20", "Bass: adoptantes (miles)", 1, [2], 12, 132, "periodo", "miles", hoja_datos=wb["Bass"], ancho=16, alto=7.5)
grafica(ws, "A36", "Pesca: diagrama de bifurcación", 6, [7, 8], 13, 38, "cuota H", "N*", hoja_datos=wb["Pesca"], ancho=16, alto=7.5)
grafica(ws, "F36", "Richardson: armas de cada país", 1, [2, 3], 17, 317, "años", "armas", hoja_datos=wb["Richardson"], ancho=16, alto=7.5)

ws = wb.create_sheet("Inicio", 0); ws.sheet_view.showGridLines = False; ws.column_dimensions["A"].width = 22; ws.column_dimensions["B"].width = 90
ws["A1"] = "Dinámica de sistemas y aplicaciones · Semana 2"; ws["A1"].font = F_TIT
ws["A2"] = "Unidad Modular III. Libro de modelos vivos: cambie los parámetros y observe las gráficas y el tablero."; ws["A2"].font = F_SUB
ws["A4"] = "Cómo leer el libro"; ws["A4"].font = F_BOLD
c = ws["A5"]; c.value = "Parámetro"; c.font = F_IN; c.fill = FILL_IN; ws["B5"] = "Celdas amarillas con texto azul: valores que usted puede cambiar."
c = ws["A6"]; c.value = "Indicador"; c.font = F_KPI; c.fill = FILL_KPI; ws["B6"] = "Resultados calculados con fórmulas."
ws["A7"] = "Tabla"; ws["B7"] = "Cada fila es un paso de tiempo calculado con el método de Euler: nivel nuevo = nivel anterior + DT × (entradas − salidas)."
ws["A9"] = "Hojas"; ws["A9"].font = F_BOLD
indice = [("Tablero", "Indicadores clave y cuatro gráficas de resumen."), ("Chingaza", "Racionamiento de agua de Bogotá 2024-2025: modelo calibrado, tres políticas."), ("Deuda", "Crecimiento exponencial."), ("Tinto", "Búsqueda de objetivo y efecto del paso DT."),
    ("Bass", "Curva S de adopción de una billetera digital."), ("Ducha", "Sobrepaso por retardo (DELAY3)."), ("Petroleo", "Sobrepaso y colapso (Hubbert)."), ("Papa", "Oscilación de telaraña."), ("Richardson", "Estabilidad del equilibrio en una carrera armamentista."),
    ("Pesca", "Bifurcación silla-nodo con cuota."), ("Lago", "Catástrofe e histéresis."), ("Cadena", "Cadena de envejecimiento del personal.")]
for i, (h, d) in enumerate(indice):
    r = 10 + i; c = ws.cell(r, 1, h); c.hyperlink = f"#'{h}'!A1"; c.font = Font(name="Arial", size=10, color="3E7CB1", underline="single"); ws.cell(r, 2, d).font = F_BODY
ws["A23"] = "Fuentes"; ws["A23"].font = F_BOLD
ws["B23"] = "Alcaldía de Bogotá y EAAB (2024-2025); Bass (1969); Richardson (1960); Carpenter et al. (1999); Ezekiel (1938); Hubbert (1956); Sterman (2000)."
for row in ws["A4:B23"]:
    for c in row:
        if c.font == Font(): c.font = F_BODY
print("Tablero e índice listos. Hojas:", wb.sheetnames)

In [ ]:
ARCHIVO = "Dashboards_Dinamica_Sistemas_Semana2.xlsx"
wb.calculation.fullCalcOnLoad = True  # Excel recalcula todo al abrir
wb.save(ARCHIVO); print("Guardado:", ARCHIVO)
try:
    from google.colab import files
    files.download(ARCHIVO)
except ImportError:
    print("Fuera de Colab: el archivo quedó en la carpeta de trabajo.")

---
## 8. Ejercicios
1. En la hoja `Chingaza`, cambie la política a «Sin racionamiento» y anote los días por debajo de 20 %. ¿Qué consumo fijo mínimo logra cero días bajo 20 %?
2. Calibre de nuevo las afluencias dejando por fuera el dato del 11 de abril de 2025 y úselo para validar. ¿Cuánto error tiene la predicción?
3. En `Deuda`, ¿qué pago mensual mantiene la deuda constante? Demuéstrelo con la ecuación del nivel.
4. En `Tinto`, ¿con qué DT el error máximo supera 1 °C?
5. En `Bass`, encuentre el par (p, q) que pone el pico en el periodo 4.
6. En `Ducha`, ¿qué tiempo de ajuste elimina el sobrepaso? Relacione su respuesta con el retardo de la tubería.
7. En `Richardson`, encuentre el valor de k que vuelve inestable el sistema si l = 0,5, a = 0,6 y b = 0,7.
8. En `Pesca`, ¿qué población inicial mínima sobrevive con H = 100?
9. En `Lago`, pruebe b = 0,6. ¿Por qué el diagnóstico dice «irreversible»?
10. En `Cadena`, reemplace el recorte por un aumento al 150 % durante tres años y describa el efecto 20 años después.